# CEF 2026 — FERLD RF map report in Colab

This notebook creates a publication-ready PDF/PNG map of the FERLD Random Forest forest-loss result directly in Google Colab, without QGIS and without local computer paths. It reads the outputs generated by the main FERLD forest-loss notebook from `My Drive/Workshop/Outputs`.


## 1. Install libraries

Colab is temporary, so these packages are installed at the beginning of each session.

In [ ]:
!pip install -q rasterio geopandas matplotlib contextily shapely pyproj

## 2. Mount Google Drive and locate workshop files

Expected structure:

```text
My Drive/
└── Workshop/
    ├── Limits/
    │   └── FERLD.geojson
    └── Outputs/
        └── FERLD_RF_forest_loss_no_loss.tif
```


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

WORKDIR = Path('/content/drive/MyDrive/Workshop')
LIMITS_DIR = WORKDIR / 'Limits'
OUTPUT_DIR = WORKDIR / 'Outputs'
REPORT_DIR = OUTPUT_DIR / 'Reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

FERLD_PATH = LIMITS_DIR / 'FERLD.geojson'
RF_TIF = OUTPUT_DIR / 'FERLD_RF_forest_loss_no_loss.tif'
required = [FERLD_PATH, RF_TIF]
missing = [p for p in required if not p.exists()]

print('WORKDIR   :', WORKDIR)
print('OUTPUTS   :', OUTPUT_DIR)
print('REPORTS   :', REPORT_DIR)
print('\nFiles:')
for p in required:
    print(('OK      ' if p.exists() else 'MISSING '), p.name)

if missing:
    raise FileNotFoundError('Missing required files: ' + ', '.join(str(p) for p in missing))

## 3. Cartographic helper functions

The report uses high-contrast colors for change detection: magenta for loss, bright green for gain, and a very light stable class so the satellite basemap remains readable.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import rasterio
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import ListedColormap, BoundaryNorm, LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from rasterio.plot import plotting_extent

try:
    import contextily as ctx
    HAS_BASEMAP = True
except Exception:
    HAS_BASEMAP = False

INK = '#1f2933'
MUTED = '#5b6770'
LOSS = '#e6007e'
GAIN = '#00b894'
STABLE = '#f7f7f7'
FERLD = '#ffd43b'
BUFFER = '#42c5f5'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 12,
    'axes.labelsize': 9,
    'figure.facecolor': 'white',
})

def read_raster(path):
    src = rasterio.open(path)
    arr = src.read(1, masked=True)
    extent = plotting_extent(src)
    return src, arr, extent

def add_basemap(ax, crs):
    if not HAS_BASEMAP:
        ax.set_facecolor('#eef2f3')
        return
    try:
        ctx.add_basemap(
            ax,
            crs=crs,
            source=ctx.providers.Esri.WorldImagery,
            attribution=False,
            alpha=0.95,
        )
    except Exception as exc:
        ax.set_facecolor('#eef2f3')
        print('Basemap unavailable:', exc)

def workshop_geometries(target_crs):
    ferld_raw = gpd.read_file(FERLD_PATH).to_crs('EPSG:32618')
    try:
        ferld_geom = ferld_raw.geometry.union_all()
    except AttributeError:
        ferld_geom = ferld_raw.geometry.unary_union
    ferld = gpd.GeoDataFrame(geometry=[ferld_geom], crs='EPSG:32618').to_crs(target_crs)
    buffer = gpd.GeoDataFrame(geometry=[ferld_geom.buffer(10000)], crs='EPSG:32618').to_crs(target_crs)
    return ferld, buffer

def add_boundaries(ax, crs, show_buffer=False):
    ferld, buffer = workshop_geometries(crs)
    if show_buffer:
        buffer.boundary.plot(ax=ax, color=BUFFER, linewidth=0.9, alpha=0.55, zorder=7)
    ferld.boundary.plot(ax=ax, color=FERLD, linewidth=1.8, zorder=8)

def set_focus_extent(ax, crs, pad=0.45):
    ferld, _ = workshop_geometries(crs)
    xmin, ymin, xmax, ymax = ferld.total_bounds
    dx = xmax - xmin
    dy = ymax - ymin
    ax.set_xlim(xmin - dx * pad, xmax + dx * pad)
    ax.set_ylim(ymin - dy * pad, ymax + dy * pad)

def add_scale_bar(ax, length_km=2):
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    width = xmax - xmin
    height = ymax - ymin
    x0 = xmin + width * 0.07
    y0 = ymin + height * 0.07
    length = length_km * 1000
    ax.plot([x0, x0 + length], [y0, y0], color='white', linewidth=5, solid_capstyle='butt', zorder=20)
    ax.plot([x0, x0 + length], [y0, y0], color=INK, linewidth=2, solid_capstyle='butt', zorder=21)
    ax.text(
        x0 + length / 2,
        y0 + height * 0.018,
        f'{length_km} km',
        ha='center', va='bottom', fontsize=8, color=INK,
        path_effects=[pe.withStroke(linewidth=3, foreground='white')],
        zorder=22,
    )

def clean_axis(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
        spine.set_edgecolor('#d8dee4')

def save_figure(fig, stem):
    png = REPORT_DIR / f'{stem}.png'
    pdf = REPORT_DIR / f'{stem}.pdf'
    fig.savefig(png, dpi=220, bbox_inches='tight')
    fig.savefig(pdf, dpi=220, bbox_inches='tight')
    print('Saved:', png)
    print('Saved:', pdf)

## 4. Main RF forest-loss map

The class `0` is rendered nearly transparent. The predicted forest-loss class is rendered in magenta so it does not disappear over the satellite background.

In [ ]:
src, rf, extent = read_raster(RF_TIF)
crs = src.crs

rf_cmap = ListedColormap([
    (247/255, 247/255, 247/255, 0.08),
    (230/255, 0/255, 126/255, 0.92),
])
rf_norm = BoundaryNorm([-0.5, 0.5, 1.5], rf_cmap.N)

fig, ax = plt.subplots(figsize=(13.5, 9.2))

set_focus_extent(ax, crs, pad=0.65)
add_basemap(ax, crs)
ax.imshow(rf, extent=extent, cmap=rf_cmap, norm=rf_norm, interpolation='nearest', zorder=5)
add_boundaries(ax, crs)
add_scale_bar(ax, 2)
clean_axis(ax)
ax.set_title('Perte de couvert forestier predite — FERLD', loc='left', fontweight='bold', color=INK, pad=10, fontsize=15)
ax.text(0.0, 1.015, 'Random Forest · Sentinel-2 SR · Hansen GFC v1.11 · 2001-2023', transform=ax.transAxes, color=MUTED, fontsize=10)

legend_items = [
    Patch(facecolor=LOSS, edgecolor='none', label='Perte de foret predite RF'),
    Line2D([0], [0], color=FERLD, lw=2, label='Limite FERLD'),
]
ax.legend(handles=legend_items, loc='lower right', frameon=True, framealpha=0.92, facecolor='white', edgecolor='#d8dee4', fontsize=9)
fig.text(0.08, 0.035, 'Random Forest: 300 arbres · 8 predicteurs spectraux · 400 points par classe · precision globale ~74 %', color=MUTED, fontsize=9)
fig.text(0.08, 0.015, 'Sentinel-2 SR (ESA), 2023 · Hansen Global Forest Change v1.11 · CEF Workshop 2026 · GEE + Python', color=MUTED, fontsize=8)

save_figure(fig, 'CEF2026_carte_FERLD_colab')
plt.show()
src.close()

## Outputs

The generated report is saved in:

`My Drive/Workshop/Outputs/Reports/`

Files generated:

- `CEF2026_carte_FERLD_colab.pdf`
- `CEF2026_carte_FERLD_colab.png`
